# Computational Theory: SHA-256

This notebook works through the problems set out in the assessment brief, building up an implementation of SHA-256 as described in the [Secure Hash Standard (FIPS 180-4)](https://doi.org/10.6028/NIST.FIPS.180-4).

## Problem 0: GitHub Issues

Progress on each problem below is tracked using GitHub Issues on this repository. See the [Issues tab](../../issues) for details.

## Problem 1: Representing SHA
Problem one introduces on how how SHA can be represented through various data structures within python


### 32 Bit words 
FIPS 180-4 defines a word as a group of 32 or 64 bits depending on the SHA algorithim that is used ([FIPS 180-4, Section 2.1](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=9)).

32 bit words can be represented via two distinct data structures in python.
1. Plain int: Which has arbitrary precision, meaning it can grow as large as needed but never overflows. It will happily grow into a 33 bit number. To prevent this, you must force it to behave like a 32-bit word by explicitly discarding any bits above the 32nd (& 0xFFFFFFFF) after every single addition.([Python docs: Numeric Types](https://docs.python.org/3/builtins/stdtypes.html#numeric-types-int-float-complex))
2. Numpy: Provides uint32. Which is a native, fixed-size 32-bit unsigned integer. It features automatic hardware-level overflow. If a value exceeds 0xFFFFFFFF, it natively wraps around to 0 without requiring a manual mask.([Numpy docs: uint32](https://numpy.org/devdocs/reference/arrays.scalars.html#numpy.uint32))

In conclusion, 32 bit words would be best represented usings numpy's 32 bit data structure as SHA-256 relies heavily on modulo $2^{32}$ addition. When a value exceeds 0xFFFFFFFF, the extra bits must disappear, and the number must wrap back around to zero.

In [1]:
# import numpy under its conventionbal alias np
import numpy as np 

In [2]:
# Plain Python int: unlimited precision, so it grows past 32 bits.
x = 0xFFFFFFFF
x_plus_one = x + 1

print(f"int:    {x:#010x} + 1 = {x_plus_one:#x}")

# Masking with & 0xFFFFFFFF keeps only the lowest 32 bits.
print(f"masked: {x_plus_one & 0xFFFFFFFF:#010x}")

int:    0xffffffff + 1 = 0x100000000
masked: 0x00000000


As we can see the result above has 9 hex digits, meaning it needs 33 bits, one more than a word can hold. SHA 256 requires addition modulo $2^{32}$ ([FIPS 180-4, Section 6.2.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=27)). So with a plain `int` masking would have to be performed after each addition. The masked result, `0x00000000`, is what a true 32-bit word should give, so it's the value we expect `uint32` to produce 

In [3]:
# uint fixed size 32 bit
max_word = np.uint32(x)
# trying to overflow the 32 bit uint
y = max_word + np.uint32(1)

# print shows that its wrapped back to 0 
print(f"uint32: {max_word:#010x} + 1 = {y:#010x}")

uint32: 0xffffffff + 1 = 0x00000000


C:\Users\jhann\AppData\Local\Temp\ipykernel_29840\513019921.py:4: RuntimeWarning: overflow encountered in scalar add
  y = max_word + np.uint32(1)


The `uint32` addition seen above returns `0x00000000`. The same as the masked value from the plain `int`. No mask needed, the wrapping was done automatically. `uint32` is fixed at 32 bits, so arithmetic wraps modulo $2^{32}$, which is exactly what FIPS 180-4 requires ([Section 6.2.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=27)). The `RuntimeWarning` is encountered becuase NumPy assumes the overflow is a mistake. In SHA-256 this wrap around is intended, so the warning here can be ignored ([NumPy docs: overflow errors](https://numpy.org/doc/stable/user/basics.types.html#overflow-errors)). This is why SHA-256's 32-bit words will be represented as `numpy.uint32`